
# Yambda 16 · Two-stage ranking, validated over time

The ranker is trained on candidates/labels from the **earlier** window (`split_rank`, labels days 259–273)
and evaluated on the validation window (days 273–287), which it has never seen. No user or time
period is shared between training labels and evaluation.

Compared on validation:
* iALS alone, RRF blend (no training), ranker with all features, ranker without track-identifying features.

Needs `cands_rank_sasrec.parquet` (notebook 15) and `cands_val_sasrec.parquet` (notebook 12) in
`reports/yambda/`. ~15 min (two iALS fits + feature building).


In [1]:

import json, time
import numpy as np
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate, paired_bootstrap, artist_novelty
from musicrec.yambda import load_item_artist
from musicrec.models.baselines import MostPopular
from musicrec.models.als import IALS
from musicrec import ranking as R

pd.set_option("display.float_format", "{:,.4f}".format)
OUT = config.PROJECT_ROOT / "reports" / "yambda"
SOURCES = ["ials", "sasrec", "pop"]
K = {"ials": 200, "sasrec": 200, "pop": 100}
FULL = R.feature_columns(SOURCES)
SAFE = [f for f in FULL if not f.startswith(("item_", "artist_pop", "score_pop", "rank_pop"))]
item_artist = load_item_artist()
ials_params = json.loads((OUT / "best_ials.json").read_text())["params"]

def build(split, sasrec_file):
    t = time.time()
    ials = IALS(**ials_params).fit(split)
    pop = MostPopular(by="listeners", recent_days=7).fit(split)
    sas = pd.read_parquet(OUT / sasrec_file)
    src = {"ials": R.candidates_from_model(ials, split, K["ials"]),
           "sasrec": sas[sas["rank"] <= K["sasrec"]],
           "pop": R.candidates_from_model(pop, split, K["pop"])}
    c = R.add_features(R.add_labels(R.merge_candidates(src), split), split, item_artist, SOURCES)
    c["rrf"] = R.rrf_scores(c, SOURCES)
    print(f"{len(c):,} candidate rows in {time.time() - t:.0f}s")
    return c

def evaluate_cols(c, split, cols):
    out = {}
    for col, name in cols:
        d = c[["user_idx", "item_idx", col]].dropna()
        out[name] = evaluate(R.PrecomputedScores(d, split.n_items, name, col), split, return_per_user=True)
    return out

def table(res):
    return pd.DataFrame({n: {k: v for k, v in r.items() if k != "per_user"} for n, r in res.items()}).T[
        ["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10"]]


## Build candidates: ranker-training window and validation window

In [2]:

rank = Split.load(config.YAMBDA_SPLIT_RANK_DIR)
c_rank = build(rank, "cands_rank_sasrec.parquet")
val = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
c_val = build(val, "cands_val_sasrec.parquet")
pd.DataFrame({"ranker-train window": R.candidate_recall(c_rank, rank, SOURCES),
              "validation window": R.candidate_recall(c_val, val, SOURCES)})


2,301,922 candidate rows in 241s
2,408,617 candidate rows in 143s


,ranker-train window,validation window
ials,0.1173,0.1239
sasrec,0.0913,0.0931
pop,0.0476,0.0445
union,0.1801,0.1852
candidates per user,422.3710,424.2017


## Train on the earlier window, evaluate on validation

In [3]:

rankers = {}
for name, feats in [("full", FULL), ("safe", SAFE)]:
    t = time.time()
    rankers[name] = R.fit_ranker(c_rank, feats)
    c_val[f"ltr_{name}"] = rankers[name].predict(c_val[feats])
    print(f"ranker [{name}] trained in {time.time() - t:.0f}s, {rankers[name].best_iteration_} trees")

res = evaluate_cols(c_val, val, [("score_ials", "iALS alone"), ("rrf", "RRF blend"),
                                 ("ltr_safe", "Ranker, no track features"), ("ltr_full", "Ranker, all features")])
table(res)


ranker [full] trained in 12s, 10 trees
ranker [safe] trained in 3s, 23 trees


,recall@10,ndcg@10,hit_rate@10,recall@50,coverage@10,avg_pop@10
iALS alone,0.0346,0.0322,0.1671,0.0686,0.0549,832.0774
RRF blend,0.0387,0.0353,0.1786,0.0740,0.0568,"1,104.2816"
"Ranker, no track features",0.0377,0.0346,0.1756,0.0743,0.0857,747.0543
"Ranker, all features",0.0455,0.0437,0.2138,0.0839,0.0616,720.7965


In [4]:

base = res["iALS alone"]["per_user"]
pd.DataFrame({f"{n} vs iALS": paired_bootstrap(base, res[n]["per_user"])
              for n in ["RRF blend", "Ranker, no track features", "Ranker, all features"]}).T[
    ["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]]


,a,b,diff,rel_diff,ci_low,ci_high,p(b<=a)
RRF blend vs iALS,0.0346,0.0387,0.0041,0.1184,0.0016,0.0068,0.0005
"Ranker, no track features vs iALS",0.0346,0.0377,0.0031,0.0902,0.0007,0.0057,0.0060
"Ranker, all features vs iALS",0.0346,0.0455,0.0109,0.3141,0.0079,0.0140,0.0000


In [5]:

best_name = max(["Ranker, no track features", "Ranker, all features"], key=lambda n: res[n]["recall@10"])
pd.Series(paired_bootstrap(res["RRF blend"]["per_user"], res[best_name]["per_user"]), name=f"{best_name} vs RRF")


a          0.0387
b          0.0455
diff       0.0068
rel_diff   0.1749
ci_low     0.0037
ci_high    0.0098
p(b<=a)    0.0000
Name: Ranker, all features vs RRF, dtype: float64

In [6]:

# Familiar vs. new artists
new_artist = artist_novelty(val, item_artist)
seg = {}
for col, name in [("score_ials", "iALS alone"), ("rrf", "RRF blend"), ("ltr_full", "Ranker, all features")]:
    m = R.PrecomputedScores(c_val[["user_idx", "item_idx", col]].dropna(), val.n_items, name, col)
    seg[name] = {lab: evaluate(m, val, targets=val.targets[mask])["recall@10"]
                 for lab, mask in [("known-artist", ~new_artist), ("new-artist", new_artist)]}
pd.DataFrame(seg).T


,known-artist,new-artist
iALS alone,0.0410,0.0160
RRF blend,0.0380,0.0226
"Ranker, all features",0.0590,0.0161


In [7]:

imp = pd.Series(rankers["full"].booster_.feature_importance("gain"), index=FULL)
(imp / imp.sum()).sort_values(ascending=False).round(3)


item_age_days          0.1460
rank_ials              0.1100
item_pop_30d           0.1020
item_trend             0.0900
user_artist_plays      0.0770
user_artist_plays_7d   0.0520
score_ials             0.0480
item_pop_7d            0.0420
score_pop              0.0410
score_sasrec           0.0400
artist_pop_7d          0.0390
n_sources              0.0380
rank_pop               0.0340
item_pop_all           0.0320
rank_sasrec            0.0310
user_events_log        0.0300
user_events_7d_log     0.0260
user_days_since_last   0.0210
dtype: float64

In [8]:

table(res).to_csv(OUT / "ranker_temporal_val.csv")



## Test (run once, after we agree on the validation result)

Ranker retrained on the validation window (labels days 273–287) with the feature set chosen above,
applied to test candidates (models trained on days < 287). Set `CHOSEN` before flipping `RUN_TEST`.


In [10]:

RUN_TEST = True
CHOSEN = "full"          # or "safe", decided from the validation table above
if RUN_TEST:
    feats = FULL if CHOSEN == "full" else SAFE
    test = Split.load(config.YAMBDA_SPLIT_TEST_DIR)
    c_test = build(test, "cands_test_sasrec.parquet")
    final_ranker = R.fit_ranker(c_val, feats)
    c_test["ltr"] = final_ranker.predict(c_test[feats])
    res_t = evaluate_cols(c_test, test, [("score_ials", "iALS alone"), ("rrf", "RRF blend"), ("ltr", "Two-stage ranker")])
    base_t = res_t["iALS alone"]["per_user"]
    sig_t = pd.DataFrame({f"{n} vs iALS": paired_bootstrap(base_t, res_t[n]["per_user"]) for n in ["RRF blend", "Two-stage ranker"]}).T
    table(res_t).to_csv(OUT / "final_test_ranker.csv"); sig_t.to_csv(OUT / "final_test_ranker_significance.csv")
    display(table(res_t), sig_t[["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]])


2,672,405 candidate rows in 293s


,recall@10,ndcg@10,hit_rate@10,recall@50,coverage@10,avg_pop@10
iALS alone,0.0334,0.0300,0.1514,0.0661,0.0564,882.0132
RRF blend,0.0349,0.0312,0.1648,0.0703,0.0592,"1,167.5504"
Two-stage ranker,0.0490,0.0452,0.2090,0.0825,0.0699,663.2747


,a,b,diff,rel_diff,ci_low,ci_high,p(b<=a)
RRF blend vs iALS,0.0334,0.0349,0.0015,0.0460,-0.0008,0.0038,0.1015
Two-stage ranker vs iALS,0.0334,0.0490,0.0156,0.4679,0.0126,0.0186,0.0000
